# 13 · KPIs, signals, and the difference

You have a warehouse. Eight pipelines fill it and they all say `ok`.

**That is not the same as the numbers being right**, and notebook 9 proved it:
a field moved upstream, nothing failed, no row count changed, and a number
quietly stopped being true.

So somebody has to watch the numbers. This notebook is about what to watch, and
the distinction that makes the difference between alerting that works and
alerting everybody ignores.

![](img/oncall-2-kpi-vs-signal.png)

In [ ]:
import sys; sys.path.insert(0, '..')
from nb import show, sql, fetch, run          # the same helpers as notebooks 1 to 12

---

## The distinction, in one cell

People use these words interchangeably and then cannot work out why their
alerting is useless.

In [ ]:
import psycopg
from pipelines.lib.config import dsn, SCHEMA

# A KPI: a number with a definition. That is all it is.
with psycopg.connect(dsn()) as c:
    revenue = float(c.execute(
        f'SELECT revenue FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC LIMIT 1'
    ).fetchone()[0])          # NUMERIC comes back as Decimal, which will not
                             # subtract from a float. Cast at the boundary.

print(f'revenue yesterday: {revenue:,.2f}')
print()
print('Is that good or bad?')

**You cannot answer that**, and neither can any alerting system, because a
number on its own has no opinion about itself.

Now give it a baseline.

In [ ]:
import statistics

with psycopg.connect(dsn()) as c:
    history = [float(r[0]) for r in c.execute(
        f'SELECT revenue FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC OFFSET 1')]

baseline = statistics.mean(history)
spread   = statistics.pstdev(history)

print(f'yesterday      {revenue:>14,.2f}')
print(f'normally       {baseline:>14,.2f}   over {len(history)} days')
print(f'usual spread   {spread:>14,.2f}')
print()
print(f'z score        {(revenue - baseline) / max(spread, 0.01):>14.2f}')

> **A KPI is a number with a definition and an owner. It can never be wrong and
> it can never fire.**
>
> **A signal is a KPI plus a baseline plus a tolerance. That is the thing that
> can breach.**

The baseline is the part nobody wants to write, and it is the only part that
turns a number into something you can act on.

---

## Two kinds of baseline, and both are correct

Some numbers **have a history**. Some have **a rule**.

In [ ]:
with psycopg.connect(dsn()) as c:
    held = c.execute(f'SELECT count(*) FROM {SCHEMA}.quarantine').fetchone()[0]

print(f'records held: {held:,}')
print()
print('What is the "normal" number of held records?')
print('It is not an average of the last seven days. It is zero.')
print('Zero held records is not a statistic, it is a rule somebody decided.')

| | computed from history | decided by a person |
|---|---|---|
| example | `rides_per_day`, `revenue_per_day` | `records_held`, `pipelines_failing` |
| baseline | the mean of previous readings | a number in the code |
| breach when | it is N spreads from normal | it is anything other than the rule |
| catches | a feed that stopped | a contract refusing things |
| gets wrong | a genuinely quiet Sunday | nothing, but it needs a person to set it |

A system that only has the first kind cannot express "this should be zero". A
system that only has the second cannot express "quieter than a normal Tuesday".

---

## Writing one down properly

Four fields, and the fourth is the one people skip.

In [ ]:
from signal_service.kpis import KPI, CATALOGUE, get

k = get('surge_coverage_pct')

print(f'name       {k.name}')
print(f'title      {k.title}')
print(f'owner      {k.owner}          <- a team that exists')
print(f'watches    {k.watches}')
print(f'unit       {k.unit!r}')
print(f'judged by  {k.judgement}, baseline {k.baseline}, tolerance {k.tolerance}')
print(f'\nmeans:\n  {k.means}')

### The owner is the field that decides whether any of this was worth building

A breach with no name attached is a number on a screen that everybody assumes
somebody else is looking at.

`means` is written for that owner, who will read it at 3am having never seen
this code. Not for you, and not for the model.

### And read the tolerance

`surge_coverage_pct` has a baseline of 100 and a tolerance of **1**, not 5.

That is deliberate. In normal operation this KPI is exactly 100.0, because the
pipeline **holds** a record it cannot read a surge value from rather than
writing a null. So any drop at all is a real change, and a generous tolerance
would let a release move a field and stay under the bar until it had affected
weeks of rides.

---

## The catalogue

Thirteen KPIs, in four groups, because between them they answer the four
questions that actually go wrong.

In [ ]:
groups = {
    'did the work happen':  ('pipelines_failing', 'warehouse_lag_hours', 'records_held'),
    'is the volume right':  ('rides_per_day', 'revenue_per_day', 'events_per_ride'),
    'is the shape right':   ('completion_rate', 'cancellation_rate', 'avg_fare'),
    'is anything missing':  ('surge_coverage_pct', 'fare_coverage_pct',
                             'zone_coverage_pct', 'settlement_coverage_pct'),
}
for question, names in groups.items():
    print(f'\n{question.upper()}')
    for name in names:
        k = get(name)
        how = 'a rule' if k.judgement == 'fixed' else 'history'
        print(f'   {k.name:26} {k.owner:14} {how:8} {k.severity}')

### The one that is not about data at all

`pipelines_failing` reads `teach.runs`. It does not look at a single row of
business data. It asks **whether the work happened.**

**A pipeline that never ran leaves perfectly valid rows behind.** Every value is
correct. Every value is old. No check that looks only at values will ever
notice, which is why this one exists.

---

## Measuring, and judging, are different steps

They fail for different reasons, so keeping them apart lets you tell the
difference between "the KPI could not be measured" and "the KPI was measured and
it is wrong". Only the second is worth waking somebody for.

In [ ]:
from signal_service import evaluate as ev

reading, verdict = ev.evaluate(get('surge_coverage_pct'))

print('READING   (running the query)')
print(f'   value    {reading.value}')
print(f'   measured {reading.measured}')
print(f'   error    {reading.error}')
print()
print('VERDICT   (comparing it to normal)')
print(f'   baseline {verdict.baseline}')
print(f'   breached {verdict.breached}')
print(f'   detail   {verdict.detail}')

## Now the whole board

In [ ]:
rows = ev.evaluate_all()

print(f'{"":8} {"kpi":26} {"value":>13} {"normally":>13}   detail')
print('-' * 100)
for kpi, reading, v in rows:
    if not reading.measured:
        print(f'{"----":8} {kpi.name:26} {"not measurable":>13}   {reading.error[:40]}')
        continue
    flag = 'BREACH' if v.breached else '  ok  '
    print(f'{flag:8} {kpi.name:26} {v.value:>13,.2f} {v.baseline:>13,.2f}   {v.detail[:44]}')

---

## One bug worth showing, because it will happen to you

The first version of `evaluate_all` shared one database connection across all
thirteen KPIs, which is right: thirteen connections every minute forever is
waste.

But a failing query **aborts the whole transaction in Postgres**, and every KPI
after it died with `current transaction is aborted`. One bad query silenced
twelve good numbers.

In [ ]:
import inspect
src = inspect.getsource(ev.read)
start = src.index('except Exception as e:')
print(src[start:start + 900])

`conn.rollback()` is the whole fix. **One poison KPI must not block the rest**,
which is the same rule as one poison message on a topic.

## What you learned

- **A KPI cannot fire. A signal can.** The difference is a baseline somebody wrote
- Two kinds of baseline: **computed from history**, and **decided by a person**
- The **owner** is the field that makes a breach somebody's rather than nobody's
- A tolerance should reflect **what the number does when nothing is wrong**
- **Measuring and judging are separate steps**, because they fail differently
- One failing query can take out the other twelve. Roll back and carry on